In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_06_accounts_attributes
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_06_accounts_attributes
# Spec Ref  : §1.9 accounts_attributes (account_id, is_paying, is_excluded, mrr)
# Purpose   : Create one attributes row for EVERY account.
#             Spec DQ gate: SELECT COUNT(*) FROM accounts_all a
#               LEFT JOIN accounts_attributes aa ON a.id = aa.account_id
#               WHERE aa.account_id IS NULL  ← must return 0
# Run after : map_02 (accounts_all)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
tenant_id   = tenant_id_from_customer(customer_id)
print(f"=== Map 06: accounts_attributes ===  customer={customer_id}  tenant={tenant_id}")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"

import sys, os
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
if os.path.abspath(project_root) not in sys.path:
    sys.path.append(os.path.abspath(project_root))

from utils.pipeline_metrics import PipelineMetrics
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "02b_map_06_accounts_attributes",
    task_key       = "run_job_D_silver_map",
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()

initialize_audit_tables()

In [0]:
# CELL 3
create_map_table(
    f"{sv}.accounts_attributes",
    """
        account_id   STRING NOT NULL,
        tenant       BIGINT,
        is_paying    BOOLEAN,
        is_excluded  BOOLEAN,
        mrr          DOUBLE
    """,
    cluster_by="account_id"
)

In [0]:
# CELL 4 ── Incremental MERGE for accounts_attributes (tenant-scoped)
# Insert-only: new accounts get default values. Existing accounts are NOT overwritten
# (downstream enrichment/scoring may have updated is_paying, mrr, etc.).
# Removed accounts are deleted.
try:
    safe_drop_view(f"{sv}.accounts_attributes")

    spark.sql(f"""
        MERGE INTO {sv}.accounts_attributes AS target
        USING (
            SELECT id AS account_id, tenant
            FROM {sv}.accounts_all
            WHERE tenant = {tenant_id}
        ) AS source
        ON target.account_id = source.account_id AND target.tenant = source.tenant
        WHEN NOT MATCHED THEN INSERT (account_id, tenant, is_paying, is_excluded, mrr)
            VALUES (source.account_id, source.tenant, false, false, 0.0)
        WHEN NOT MATCHED BY SOURCE AND target.tenant = {tenant_id} THEN DELETE
    """)

    n = cnt(f"{sv}.accounts_attributes")
    print(f"  accounts_attributes: {n:,} rows")
except Exception as e:
    print(f"\u274c accounts_attributes build failed: {e}")
    log_audit(
        job_name       = "02b_map_06_accounts_attributes",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# CELL 5 -- Spec DQ gate: every account must have an attributes row
try:
    uncovered = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.accounts_all a
        LEFT JOIN {sv}.accounts_attributes aa ON a.id = aa.account_id
        WHERE aa.account_id IS NULL
    """).collect()[0][0]

    n = cnt(f"{sv}.accounts_attributes")
    print(f"  accounts_attributes: {n:,} rows")
    status = '✅ PASS' if uncovered == 0 else '❌ FAIL'
    print(f"  Uncovered accounts: {uncovered}  (spec gate = 0)  {status}")
except Exception as e:
    print(f"❌ DQ verification failed: {e}")
    log_audit(
        job_name       = "02b_map_06_accounts_attributes",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "DQ_CHECK_FAILED",
        error_type     = type(e).__name__,
        error_reason   = f"DQ check failed: {str(e)[:450]}",
    )
    raise

In [0]:
try:
    total_rows = spark.table(f"{sv}.accounts_attributes").count()
    pm.save(status="SUCCESS", rows_processed=total_rows)
except Exception as e:
    print(f"❌ Metrics save failed: {e}")
    log_audit(
        job_name       = "02b_map_06_accounts_attributes",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise